# Hierarchical Model 
### Stage 1:
#### XGBoost (binary)

### Stage 2:
#### Random Forest (multiclass)

##### Resampling (undersampling + SMOTE).
##### Compare against Flat Model.

### Studies the Hierarchical impact.

In [1]:
# Evaluation metrics

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = []

def evaluate_model(y_true, y_pred, model_name):

    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro"),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted")
    }

In [2]:
# Binary classification: Attack vs Normal

import sklearn
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import pandas as pd
import numpy as np


data = pd.read_csv('UNSW_NB15.csv')
data = data.sample(n=250000, random_state=42)


# One hot encodin
# Columns that contain categorical data
categorical_cols = ['proto', 'service','state']  

# Perform one-hot encoding for categorical columns
data_encoded = pd.get_dummies(data, columns=categorical_cols)

# Spliting the encoded data into features and target
features_encoded = data_encoded.drop(['attack_cat', 'label'], axis=1)
target = data_encoded['attack_cat']  
print(target.value_counts())

# Convert multi-class to binary
y_binary = (target != 'Normal').astype(int)

# Spliting the encoded data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(features_encoded, y_binary, test_size=0.2, random_state=42)


neg = np.sum(y_train == 0)
pos = np.sum(y_train == 1)

scale_pos_weight = neg / pos
print("scale_pos_weight:", scale_pos_weight)

xgb_binary = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_binary.fit(X_train, y_train)

y_pred = xgb_binary.predict(X_test)
y_prob = xgb_binary.predict_proba(X_test)[:, 1]

results.append(
    evaluate_model(y_test, y_pred, "Binary Classification: XGBoost")
)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nROC-AUC:", roc_auc_score(y_test, y_prob))

attack_cat
Normal            90254
Generic           57073
Exploits          43175
Fuzzers           23580
DoS               15834
Reconnaissance    13580
Analysis           2607
Backdoor           2262
Shellcode          1465
Worms               170
Name: count, dtype: int64
scale_pos_weight: 0.5642597922662996
Confusion Matrix:
[[17994   116]
 [  531 31359]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     18110
           1       1.00      0.98      0.99     31890

    accuracy                           0.99     50000
   macro avg       0.98      0.99      0.99     50000
weighted avg       0.99      0.99      0.99     50000


ROC-AUC: 0.999342926982402


In [4]:
# XGBoost + RF

import sklearn
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd
import numpy as np


data = pd.read_csv('UNSW_NB15.csv')
data = data.sample(n=250000, random_state=42)


# One hot encodin
# Use the appropriate columns that contain categorical data
categorical_cols = ['proto', 'service','state']  

# Perform one-hot encoding for categorical columns
data_encoded = pd.get_dummies(data, columns=categorical_cols)

# Spliting the encoded data into features and target
features_encoded = data_encoded.drop(['attack_cat', 'label'], axis=1)
target = data_encoded['attack_cat']  
print(target.value_counts())

# Convert multi-class to binary
# y_binary = (target != 'Normal').astype(int)
target_binary = data_encoded['label']

# Split the encoded data into training and testing sets
X_train, X_test, y_binary_train, y_binary_test, y_multi_train, y_multi_test = train_test_split(
    features_encoded,
    target_binary,
    target,              # original attack_cat
    test_size=0.2,
    stratify=target_binary,
    random_state=42
)


neg = np.sum(y_binary_train == 0)
pos = np.sum(y_binary_train == 1)

scale_pos_weight = neg / pos
print("scale_pos_weight:", scale_pos_weight)
xgb_binary = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_binary.fit(X_train, y_binary_train)

# Stage 1 predictions
y_stage1_pred = xgb_binary.predict(X_test)
y_prob = xgb_binary.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_binary_test, y_stage1_pred))

print("\nClassification Report:")
print(classification_report(y_binary_test, y_stage1_pred))

print("\nROC-AUC:", roc_auc_score(y_binary_test, y_prob))


# Keep only real attack samples
# Now only need the attack samples from the training set where target_binary == 1 
attack_train_idx = y_binary_train[y_binary_train == 1].index
print("These are the indexes where attacks rows are: ", attack_train_idx)
X_train_attack = X_train.loc[attack_train_idx]
y_train_attack = y_multi_train.loc[attack_train_idx]
print("Attacks without normal: ", y_train_attack)

rf_attack = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_attack.fit(X_train_attack, y_train_attack)

# Prepare final prediction array
attack_indices = np.where(y_stage1_pred == 1)[0]

final_predictions = pd.Series(['Normal'] * len(X_test), index=X_test.index)

if len(attack_indices) > 0:
    attack_preds = rf_attack.predict(X_test.iloc[attack_indices])
    
    # Convert positional indices → real index labels
    real_indices = X_test.index[attack_indices]

    final_predictions.loc[real_indices] = attack_preds


final_predictions = pd.Series(final_predictions)

results.append(
    evaluate_model(y_multi_test, final_predictions, "Hierarchical XGB + RF")
)

print(confusion_matrix(y_multi_test, final_predictions))
print(classification_report(y_multi_test, final_predictions))



attack_cat
Normal            90254
Generic           57073
Exploits          43175
Fuzzers           23580
DoS               15834
Reconnaissance    13580
Analysis           2607
Backdoor           2262
Shellcode          1465
Worms               170
Name: count, dtype: int64
scale_pos_weight: 0.5649819635828697
Confusion Matrix:
[[17948   103]
 [  554 31395]]

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98     18051
           1       1.00      0.98      0.99     31949

    accuracy                           0.99     50000
   macro avg       0.98      0.99      0.99     50000
weighted avg       0.99      0.99      0.99     50000


ROC-AUC: 0.9992670354691567
These are the indexes where attacks rows are:  Index([148074, 168195,  53377, 191575, 135062, 181887, 187953, 156170,  94865,
        60977,
       ...
        87155, 187196, 138864, 184681, 193303, 112987, 156479, 167972, 150862,
       235203],
      dtyp

In [5]:
# Optuna Optimazation 

import sklearn
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
from sklearn.metrics import f1_score
import optuna
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import pandas as pd


data = pd.read_csv('UNSW_NB15.csv')
data = data.sample(n=250000, random_state=42)


# One hot encodin
# Use the appropriate columns that contain categorical data
categorical_cols = ['proto', 'service','state']  # Replace with your categorical column names

# Performing one-hot encoding for categorical columns
data_encoded = pd.get_dummies(data, columns=categorical_cols)

# Spliting the encoded data into features and target
features_encoded = data_encoded.drop(['attack_cat', 'label'], axis=1)
target = data_encoded['attack_cat'] 
print(target.value_counts())

# Convert multi-class to binary
# y_binary = (target != 'Normal').astype(int)
target_binary = data_encoded['label']

# Spliting the encoded data into training and testing sets
X_train, X_test, y_binary_train, y_binary_test, y_multi_train, y_multi_test = train_test_split(
    features_encoded,
    target_binary,
    target,              # original attack_cat
    test_size=0.2,
    stratify=target_binary,
    random_state=42
)


neg = np.sum(y_binary_train == 0)
pos = np.sum(y_binary_train == 1)

scale_pos_weight = neg / pos
print("scale_pos_weight:", scale_pos_weight)
xgb_binary = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    scale_pos_weight=scale_pos_weight,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_binary.fit(X_train, y_binary_train)

# Stage 1 predictions
probs = xgb_binary.predict_proba(X_test)[:, 1]
y_stage1_pred = (probs > 0.3).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_binary_test, y_stage1_pred))

print("\nClassification Report:")
print(classification_report(y_binary_test, y_stage1_pred))

print("\nROC-AUC:", roc_auc_score(y_binary_test, probs))


# Keep only real attack samples
# Now only need the attack samples from the training set where target_binary == 1 
attack_train_idx = y_binary_train[y_binary_train == 1].index
if len(attack_train_idx) == 0:
    raise ValueError("No attack samples found in training set")
print("These are the indexes where attacks rows are: ", attack_train_idx)
X_train_attack = X_train.loc[attack_train_idx]
y_train_attack = y_multi_train.loc[attack_train_idx]
print("Attacks without normal: ", y_train_attack)


def objective(trial):
    target_count = trial.suggest_int("target_count", 1000, 15000)
    n_estimators = trial.suggest_int("n_estimators", 100, 500)
    max_depth = trial.suggest_int("max_depth", 5, 30)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_scores = []

    for train_idx, val_idx in skf.split(X_train_attack, y_train_attack):

        X_tr = X_train_attack.iloc[train_idx]
        y_tr = y_train_attack.iloc[train_idx]

        X_val_fold = X_train_attack.iloc[val_idx]
        y_val_fold = y_train_attack.iloc[val_idx]

        # Compute counts from THIS fold only
        counts = y_tr.value_counts()

        undersample_dict = {
            cls: min(count, target_count)
            for cls, count in counts.items()
        }

        # UnderSampling 
        rus = RandomUnderSampler(
            sampling_strategy=undersample_dict,
            random_state=42
        )

        X_res, y_res = rus.fit_resample(X_tr, y_tr)

        # SMOTE 
        smote = SMOTE(
            
            sampling_strategy="not majority",
            random_state=42
        )

        X_res, y_res = smote.fit_resample(X_res, y_res)

        # RandomForest
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_res, y_res)

        preds = model.predict(X_val_fold)

        fold_scores.append(
            f1_score(y_val_fold, preds, average="macro")
        )

    return np.mean(fold_scores)


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)
best_trial = study.best_trial
print("Best target_count:", study.best_params)
print("Best F1:", study.best_value)
print("Best trial:", best_trial)

# Retrain using new best params
best_target_count = study.best_params["target_count"]
best_n_estimators = study.best_params["n_estimators"]
best_max_depth = study.best_params["max_depth"]
best_min_samples_split = study.best_params["min_samples_split"]

# Under-sample
counts = y_train_attack.value_counts()

undersample_dict = {
    cls: min(count, best_target_count)
    for cls, count in counts.items()
}

rus = RandomUnderSampler(
    sampling_strategy=undersample_dict,
    random_state=42
)

X_res, y_res = rus.fit_resample(X_train_attack, y_train_attack)

# SMOTE
cat_features = ["proto", "service", "state"]

cat_indices = [
    X_train_attack.columns.get_loc(col)
    for col in X_train_attack.columns
    if col in cat_features
]

smote = SMOTE(
    sampling_strategy="not majority",
    random_state=42
)

X_res, y_res = smote.fit_resample(X_res, y_res)

rf_attack = RandomForestClassifier(
    n_estimators=best_n_estimators,
    max_depth=best_max_depth,
    min_samples_split=best_min_samples_split,
    random_state=42,
    n_jobs=-1
)

rf_attack.fit(X_res, y_res)

optuna.visualization.plot_optimization_history(study)

# Prepare final prediction array
attack_indices = np.where(y_stage1_pred == 1)[0]

final_predictions = pd.Series(['Normal'] * len(X_test), index=X_test.index)

if len(attack_indices) > 0:

    # Compute attack predictions
    attack_preds = rf_attack.predict(X_test.iloc[attack_indices])

    # Assign predictions
    final_predictions.iloc[attack_indices] = attack_preds


final_predictions = pd.Series(final_predictions)

results.append(
    evaluate_model(y_multi_test, final_predictions, "Hierarchical XGB + RF Resampled")
)



print(confusion_matrix(y_multi_test, final_predictions))
print(classification_report(y_multi_test, final_predictions))



C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


attack_cat
Normal            90254
Generic           57073
Exploits          43175
Fuzzers           23580
DoS               15834
Reconnaissance    13580
Analysis           2607
Backdoor           2262
Shellcode          1465
Worms               170
Name: count, dtype: int64
scale_pos_weight: 0.5649819635828697
Confusion Matrix:
[[17768   283]
 [  333 31616]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     18051
           1       0.99      0.99      0.99     31949

    accuracy                           0.99     50000
   macro avg       0.99      0.99      0.99     50000
weighted avg       0.99      0.99      0.99     50000


ROC-AUC: 0.9992670354691567
These are the indexes where attacks rows are:  Index([148074, 168195,  53377, 191575, 135062, 181887, 187953, 156170,  94865,
        60977,
       ...
        87155, 187196, 138864, 184681, 193303, 112987, 156479, 167972, 150862,
       235203],
      dtyp

[I 2026-03-06 11:36:52,134] A new study created in memory with name: no-name-9bd692f1-7d44-4ea1-91ef-5bc244c3179c
[I 2026-03-06 11:37:30,048] Trial 0 finished with value: 0.5219136097348398 and parameters: {'target_count': 3762, 'n_estimators': 330, 'max_depth': 27, 'min_samples_split': 2}. Best is trial 0 with value: 0.5219136097348398.
[I 2026-03-06 11:37:54,261] Trial 1 finished with value: 0.5142397536522401 and parameters: {'target_count': 2459, 'n_estimators': 399, 'max_depth': 23, 'min_samples_split': 7}. Best is trial 0 with value: 0.5219136097348398.
[I 2026-03-06 11:38:10,515] Trial 2 finished with value: 0.5124738991776031 and parameters: {'target_count': 2355, 'n_estimators': 265, 'max_depth': 19, 'min_samples_split': 10}. Best is trial 0 with value: 0.5219136097348398.
[I 2026-03-06 11:39:02,890] Trial 3 finished with value: 0.47862425518615215 and parameters: {'target_count': 13465, 'n_estimators': 225, 'max_depth': 8, 'min_samples_split': 6}. Best is trial 0 with value: 

Best target_count: {'target_count': 10097, 'n_estimators': 127, 'max_depth': 30, 'min_samples_split': 5}
Best F1: 0.5595419979499077
Best trial: FrozenTrial(number=11, state=<TrialState.COMPLETE: 1>, values=[0.5595419979499077], datetime_start=datetime.datetime(2026, 3, 6, 11, 41, 44, 469693), datetime_complete=datetime.datetime(2026, 3, 6, 11, 42, 29, 671928), params={'target_count': 10097, 'n_estimators': 127, 'max_depth': 30, 'min_samples_split': 5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'target_count': IntDistribution(high=15000, log=False, low=1000, step=1), 'n_estimators': IntDistribution(high=500, log=False, low=100, step=1), 'max_depth': IntDistribution(high=30, log=False, low=5, step=1), 'min_samples_split': IntDistribution(high=10, log=False, low=2, step=1)}, trial_id=11, value=None)
[[  112   140   255    24     7     0    15     3     0     0]
 [   77   118   194    27     9     0     1     9     5     0]
 [  178   311  1970   453    53     

In [6]:
import pandas as pd

comparison_table = pd.DataFrame(results)
comparison_table = comparison_table.round(4)

print(comparison_table)

                             Model  Accuracy  Macro Precision  Macro Recall  \
0   Binary Classification: XGBoost    0.9871           0.9838        0.9885   
1            Hierarchical XGB + RF    0.8637           0.6437        0.5895   
2  Hierarchical XGB + RF Resampled    0.8402           0.5694        0.6745   

   Macro F1  Weighted F1  
0    0.9861       0.9871  
1    0.6093       0.8623  
2    0.5855       0.8519  
